# Concurrency & Async

Python offers three concurrency models. Choosing the right one depends on your bottleneck.

| Model | Best for | GIL impact |
|---|---|---|
| `threading` | I/O-bound (network, file) | Released during I/O |
| `asyncio` | Many concurrent I/O tasks | Single thread, cooperative |
| `multiprocessing` | CPU-bound computation | Bypassed (separate processes) |

**In this notebook:**
- `threading.Thread` basics
- Thread safety with `Lock`
- `ThreadPoolExecutor` + `as_completed`
- `asyncio` — coroutines, `await`, `gather`
- `asyncio.Queue` — producer/consumer
- `asyncio.Semaphore` — rate limiting
- `multiprocessing.Pool` — CPU-bound work

## 1. threading.Thread

Threads share memory and run concurrently. Python's GIL allows only one thread to run Python bytecode at a time, but I/O operations release it — so threading helps for I/O-bound tasks.

In [ ]:
import threading
import time

def worker(name, delay):
    print(f'{name} starting')
    time.sleep(delay)     # releases the GIL → other threads can run
    print(f'{name} done after {delay}s')

t1 = threading.Thread(target=worker, args=('Alpha', 0.3))
t2 = threading.Thread(target=worker, args=('Beta',  0.1))
t3 = threading.Thread(target=worker, args=('Gamma', 0.2))

start = time.perf_counter()
for t in [t1, t2, t3]: t.start()
for t in [t1, t2, t3]: t.join()   # wait for all to finish
print(f'Total: {time.perf_counter()-start:.2f}s')  # ~0.3s, not 0.6s

## 2. Thread Safety — Lock

Shared mutable state requires protection. Without a lock, concurrent read-modify-write operations produce race conditions.

In [ ]:
import threading

# Unsafe — race condition
counter_unsafe = 0
def increment_unsafe():
    global counter_unsafe
    for _ in range(10_000):
        counter_unsafe += 1

# Safe — Lock protects the critical section
counter_safe = 0
lock = threading.Lock()
def increment_safe():
    global counter_safe
    for _ in range(10_000):
        with lock:
            counter_safe += 1

threads = [threading.Thread(target=increment_safe) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Safe counter: {counter_safe}')   # always 50000

## 3. ThreadPoolExecutor

`concurrent.futures` manages a thread pool and returns `Future` objects. `as_completed()` yields futures as they finish — not in submission order.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random, time

random.seed(42)

def fetch(url):
    time.sleep(random.uniform(0.1, 0.4))  # simulate I/O
    return f'Data from {url}'

urls = [f'url_{i}' for i in range(1, 7)]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(fetch, url): url for url in urls}
    for future in as_completed(futures):
        print(future.result())
print(f'Total: {time.perf_counter()-start:.2f}s')

## 4. asyncio — Coroutines and await

`async def` defines a coroutine. `await` suspends it until the awaitable completes, allowing other coroutines to run.

> **Key rule:** `await` can only be used inside `async def` functions.

In [ ]:
import asyncio

async def fetch_data(source, delay):
    print(f'  Fetching {source}...')
    await asyncio.sleep(delay)   # non-blocking pause
    print(f'  {source} ready!')
    return {'source': source, 'data': f'result_{source}'}

async def main():
    # Sequential — waits for each before starting next
    result1 = await fetch_data('API-1', 0.3)
    result2 = await fetch_data('API-2', 0.2)
    print(result1, result2)

asyncio.run(main())

## 5. asyncio.gather — Concurrent Tasks

`gather()` runs coroutines **concurrently**. Total time ≈ max individual time (not sum).

In [ ]:
import asyncio, time

async def task(name, delay):
    await asyncio.sleep(delay)
    return f'{name}: done'

async def main():
    start = time.perf_counter()

    results = await asyncio.gather(
        task('A', 0.5),
        task('B', 0.3),
        task('C', 0.4),
    )

    print(results)
    print(f'Total: {time.perf_counter()-start:.2f}s')  # ~0.5s not 1.2s

asyncio.run(main())

## 6. asyncio.Queue — Producer / Consumer

`asyncio.Queue` is the standard handoff mechanism between async tasks. It blocks producers when full and consumers when empty — natural back-pressure.

In [ ]:
import asyncio

async def producer(queue, items):
    for item in items:
        await asyncio.sleep(0.05)
        await queue.put(item)
        print(f'  [P] produced {item}')
    await queue.put(None)   # sentinel — signals end

async def consumer(queue, name):
    while True:
        item = await queue.get()
        if item is None:
            break
        await asyncio.sleep(0.02)
        print(f'  [{name}] consumed {item}')

async def main():
    q = asyncio.Queue()
    await asyncio.gather(
        producer(q, range(1, 6)),
        consumer(q, 'C'),
    )

asyncio.run(main())

## 7. asyncio.Semaphore — Rate Limiting

`Semaphore(n)` allows at most `n` coroutines inside `async with sem` at once.

In [ ]:
import asyncio, time, random

async def limited_fetch(sem, name):
    async with sem:   # max 2 concurrent
        print(f'  Start: {name}')
        await asyncio.sleep(random.uniform(0.1, 0.3))
        print(f'  Done:  {name}')

async def main():
    random.seed(1)
    sem = asyncio.Semaphore(2)   # max 2 at a time
    jobs = [f'job_{i}' for i in range(6)]
    start = time.perf_counter()
    await asyncio.gather(*[limited_fetch(sem, j) for j in jobs])
    print(f'Total: {time.perf_counter()-start:.2f}s')

asyncio.run(main())

## 8. multiprocessing — CPU-bound Work

Multiprocessing spawns real OS processes — each with its own Python interpreter and GIL. True parallelism for CPU-intensive tasks.

> **Important:** Always guard the entry point with `if __name__ == '__main__':` on Windows.

In [ ]:
# This cell is best run as a standalone script on Windows.
# In a Jupyter notebook it works fine on Linux/macOS.

from multiprocessing import Pool
import math

def heavy(n):
    return sum(math.sqrt(i) for i in range(n))

# if __name__ == '__main__':   # required on Windows scripts
numbers = [500_000, 1_000_000, 750_000, 1_250_000]
with Pool(processes=4) as pool:
    results = pool.map(heavy, numbers)
print([round(r, 2) for r in results])

## Concurrency Model Quick Reference

```python
# Threading
t = threading.Thread(target=func, args=(a, b))
t.start(); t.join()
with threading.Lock(): ...   # mutual exclusion

# ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=N) as ex:
    futures = [ex.submit(func, arg) for arg in items]
    results = [f.result() for f in as_completed(futures)]

# asyncio
async def coro(): await something()
asyncio.run(coro())                          # entry point
await asyncio.gather(c1(), c2(), c3())       # concurrent
task = asyncio.create_task(coro())           # schedule without awaiting
await asyncio.wait_for(coro(), timeout=5)    # with timeout
async with asyncio.Semaphore(N): ...         # rate limit

# multiprocessing
with Pool(N) as p:
    results = p.map(func, items)
```

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | Thread basics, asyncio sequential vs gather |
| [02-medium.py](exercises/02-medium.py) | Medium | SafeCounter, ThreadPoolExecutor, producer/consumer, timeout |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | Semaphore batch processor, EventBus, back-pressure pipeline |

Solutions: [solutions/](solutions/)